# PaperMind — Notebook 3: Sub-Question Query Engine

Notebook 2's router picks **one** tool per query — great when each question is about a single paper. But many real questions span multiple papers:

> *"Compare the architecture of the Transformer and BERT."*

A router would have to pick one paper and miss half the answer. A naive "merged index" approach (one big shared index) blurs chunks from different papers together and produces incoherent attribution.

**`SubQuestionQueryEngine` is the right tool for cross-paper questions.** Given a complex question and a set of `QueryEngineTool`s, an LLM:

1. Decomposes the question into focused **sub-questions**, each tagged with the tool it should hit.
2. Runs each sub-question against its assigned tool — same per-paper retrieval as notebook 2.
3. Bundles all sub-answers and synthesises a final coherent response.

We re-use the two indexes built in notebook 2 (no re-embedding) and watch the decomposition happen via a callback handler that captures every sub-question / sub-answer pair.

## 1. Setup — env, async patch, LLM, embeddings

Same Groq + BGE setup as notebook 2. Sub-question decomposition fires *several* LLM calls per user query (1 generation + N sub-answers + 1 synthesis), so Groq's higher free-tier RPM matters even more here.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio

nest_asyncio.apply()

load_dotenv("../.env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "GROQ_API_KEY not found in ../.env"

from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

llm = Groq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm = llm
Settings.embed_model = embed_model

print("LLM (Groq) + embeddings configured")

LLM (Groq) + embeddings configured


## 2. Load existing indexes from disk

Notebook 2 already embedded both papers and persisted them to `../indexes/attention/` and `../indexes/bert/`. We reload from disk — no re-embedding, no API calls. If either index is missing, run notebook 2 first.

In [2]:
from llama_index.core import StorageContext, load_index_from_storage

INDEX_ROOT = Path("../indexes")
paper_dirs = {
    "attention": INDEX_ROOT / "attention",
    "bert": INDEX_ROOT / "bert",
}

indexes = {}
for name, idx_dir in paper_dirs.items():
    assert idx_dir.exists() and any(idx_dir.iterdir()), (
        f"No index at {idx_dir}. Run notebook 2 first to build it."
    )
    storage_context = StorageContext.from_defaults(persist_dir=str(idx_dir))
    indexes[name] = load_index_from_storage(storage_context)
    print(f"[{name}] loaded from {idx_dir}")

[attention] loaded from ../indexes/attention
[bert] loaded from ../indexes/bert


## 3. Wrap each index as a QueryEngineTool

Same descriptions as notebook 2 — they double as routing hints for the sub-question generator. The LLM reads these descriptions when it decomposes a complex question, so it knows which sub-question to send where.

In [3]:
from llama_index.core.tools import QueryEngineTool

attention_engine = indexes["attention"].as_query_engine(similarity_top_k=3)
bert_engine = indexes["bert"].as_query_engine(similarity_top_k=3)

attention_tool = QueryEngineTool.from_defaults(
    query_engine=attention_engine,
    name="attention_paper",
    description=(
        "The original Transformer paper, 'Attention Is All You Need' (Vaswani et al., 2017). "
        "Use for questions about: scaled dot-product attention, multi-head self-attention, "
        "encoder-decoder stacks, sinusoidal positional encoding, training data and "
        "hyperparameters of the original Transformer, and machine-translation experiments "
        "on WMT 2014 English-German / English-French."
    ),
)

bert_tool = QueryEngineTool.from_defaults(
    query_engine=bert_engine,
    name="bert_paper",
    description=(
        "The BERT paper (Devlin et al., 2018) — a bidirectional transformer encoder "
        "pre-trained on masked language modeling (MLM) and next-sentence prediction (NSP), "
        "then fine-tuned on downstream NLP tasks. Use for questions about: pre-training "
        "objectives, BookCorpus + Wikipedia training data, WordPiece tokenization, "
        "fine-tuning on GLUE / SQuAD, and BERT-Base vs BERT-Large."
    ),
)

tools = [attention_tool, bert_tool]
print("Tools ready:", [t.metadata.name for t in tools])

Tools ready: ['attention_paper', 'bert_paper']


## 4. Capture sub-questions from response.source_nodes

When `SubQuestionQueryEngine` runs, it attaches one source node per sub-question to the final response. In recent LlamaIndex versions the text of that node looks like:

```
Sub question: <generated question>
Response: <answer that tool returned>
```

We parse this with a regex in the helper below — version-stable, no callbacks needed. The remaining nodes in `response.source_nodes` are the underlying paper chunks each sub-engine retrieved; we filter them out.

(The tool name isn't embedded in the node text — if you need it, enable `verbose=True` on the engine and the live print will show `[tool_name] Q: ...` lines. We keep verbose off for clean structured output.)

In [4]:
from typing import Any, Optional, Dict, List
from llama_index.core.callbacks.base_handler import BaseCallbackHandler
from llama_index.core.callbacks import CallbackManager, CBEventType, EventPayload


class SubQuestionTracker(BaseCallbackHandler):
    """Collects every SubQuestionAnswerPair emitted during a query."""

    def __init__(self):
        super().__init__(event_starts_to_ignore=[], event_ends_to_ignore=[])
        self.pairs: List = []

    def reset(self) -> None:
        self.pairs = []

    def start_trace(self, trace_id: Optional[str] = None) -> None:
        pass

    def end_trace(
        self,
        trace_id: Optional[str] = None,
        trace_map: Optional[Dict[str, Any]] = None,
    ) -> None:
        pass

    def on_event_start(
        self,
        event_type: CBEventType,
        payload: Optional[Dict[str, Any]] = None,
        event_id: str = "",
        parent_id: str = "",
        **kwargs: Any,
    ) -> str:
        return event_id

    def on_event_end(
        self,
        event_type: CBEventType,
        payload: Optional[Dict[str, Any]] = None,
        event_id: str = "",
        **kwargs: Any,
    ) -> None:
        if event_type == CBEventType.SUB_QUESTION and payload is not None:
            sqp = payload.get(EventPayload.SUB_QUESTION)
            if sqp is not None:
                self.pairs.append(sqp)


tracker = SubQuestionTracker()
Settings.callback_manager = CallbackManager([tracker])
print("Sub-question tracker registered")

Sub-question tracker registered


## 5. Build the SubQuestionQueryEngine

`use_async=False` keeps execution sequential, so the events arrive in a deterministic order — easier to read in this notebook. With `use_async=True` you'd get parallel sub-question execution (faster), but the print order in our tracker would be interleaved.

In [5]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.question_gen.llm_generators import LLMQuestionGenerator

question_gen = LLMQuestionGenerator.from_defaults(llm=llm)

sub_q_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=tools,
    question_gen=question_gen,
    use_async=False,
    verbose=False,  # we extract sub-questions ourselves from response.source_nodes
)

## 6. Run cross-paper queries

Three questions that genuinely require both papers. For each:
1. Print the user's question.
2. Print every generated sub-question, the tool it was sent to, and that tool's answer.
3. Print the final synthesised response that combines all sub-answers.

A small `time.sleep` between queries keeps us comfortably under Groq's free-tier RPM even with the burst of LLM calls each query produces.

In [6]:
import re
import time

# SubQuestionQueryEngine emits one source node per sub-question whose text is
# of the form: "Sub question: <Q> Response: <A>". The remaining source nodes
# are the underlying paper chunks each sub-engine retrieved — we filter those out.
SUBQ_PATTERN = re.compile(r"^Sub question:\s*(?P<q>.+?)\s*Response:\s*(?P<a>.+)$", re.DOTALL)


def _node_text(n) -> str:
    try:
        return n.node.get_content()
    except Exception:
        return getattr(n.node, "text", str(n.node))


def extract_sub_questions(response):
    pairs = []
    for n in response.source_nodes:
        m = SUBQ_PATTERN.match(_node_text(n))
        if m:
            pairs.append((m.group("q").strip(), m.group("a").strip()))
    return pairs


def run_complex_query(query: str) -> None:
    print("=" * 100)
    print(f"Q: {query}\n")

    response = sub_q_engine.query(query)
    pairs = extract_sub_questions(response)

    print(f"--- Generated {len(pairs)} sub-questions ---\n")
    for i, (sub_q, ans) in enumerate(pairs, 1):
        indented = "\n       ".join(ans.splitlines()) or "(no answer)"
        print(f"[{i}] Q: {sub_q}")
        print(f"    A: {indented}\n")

    print("--- Final synthesised answer ---")
    print(response)
    print()


queries = [
    "Compare the architecture of the Transformer and BERT models.",
    "What training data and objectives were used in each model?",
    "What were the key innovations and limitations of each approach?",
]

for i, q in enumerate(queries):
    run_complex_query(q)
    if i < len(queries) - 1:
        time.sleep(3)

Q: Compare the architecture of the Transformer and BERT models.

--- Generated 2 sub-questions ---

[1] Q: What is the architecture of the Transformer model
    A: The Transformer model follows an overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. The encoder is composed of a stack of 6 identical layers, each with two sub-layers: a multi-head self-attention mechanism and a simple, position-wise fully connected feed-forward network. The decoder is also composed of a stack of 6 identical layers, with an additional third sub-layer that performs multi-head attention over the output of the encoder stack. Residual connections and layer normalization are employed around each sub-layer.

[2] Q: What is the architecture of the BERT model
    A: The BERT model is a deep bidirectional Transformer, with two versions: BERTBASE and BERTLARGE. BERTBASE contains 110M parameters, while BERTLARGE contains 340M parameters. The model 

## How decomposition works (and when to use it)

**Per query, under the hood:**
1. **Sub-question generation** — the LLM gets the user's question plus each tool's `description`. It responds with a JSON list of `(sub_question, tool_name)` pairs.
2. **Per-tool retrieval** — each sub-question hits its assigned `QueryEngineTool`, which runs the same per-paper RAG we built in notebook 2 (top-3 chunks → answer).
3. **Final synthesis** — the LLM gets the original question plus all sub-question/answer pairs and writes one coherent response.

**When sub-question decomposition beats other patterns:**
- *Comparisons* — "Compare X and Y" naturally splits into one sub-question per side.
- *Multi-faceted questions* — "Architecture, training data, and limitations of each model" splits into clean per-aspect sub-questions.
- *Cross-document synthesis* — anywhere the user wants conclusions that depend on multiple sources.

**When NOT to use it:**
- Single-source questions — adds a generation + synthesis call each, for no benefit. Use the router (notebook 2) instead.
- Latency-sensitive paths — N+2 LLM calls per query is much slower than a single call. For real-time use, consider `use_async=True` to parallelise sub-questions.

**Tuning levers:**
- Tool **descriptions** — same as the router; if the LLM is mis-routing sub-questions, sharpen these first.
- The **question generator prompt** — `from_defaults` uses LlamaIndex's standard prompt. You can pass a custom `question_gen` for domain-specific decomposition (e.g., always asking for a methods + results split for scientific questions).